# D9R9 — PaliGemma single-T4 owner execution

**PRE-RUNTIME TOOLING. No runtime result is claimed by this notebook file.**

Research Lead reviews this notebook before delivery. Use a fresh private Kaggle
Notebook, Linux x86_64, **GPU T4 x2** (one T4 is also accepted), Internet ON.
Enable the Kaggle Secret **HF_TOKEN** beforehand; the owner must already have
accepted any required model terms. SafeShift is public and cloned anonymously.
No GitHub credential is needed. Never paste credentials
into cells. No dataset attachment is needed.

Run all cells through the Internet-OFF barrier. The online phase proceeds
automatically; after provision it stops at the explicit attestation. Turn Internet
OFF without restarting the session, change the single attestation boolean, and
run that cell. No per-phase reporting to Codex is needed.

Execution checkout is always `180c0623149bbc7d64f5b659f6c044c39b1e1cd0`,
not the notebook's delivery commit. Model `google/paligemma-3b-mix-448`, revision
`ead2d9a35598cb89119af004f5d023b311d1c4a1`. Exact D9R8 scripts and plan are reused.
Setup/weights stay in a dedicated workspace; an existing workspace is refused.
Any phase failure stops scientific work, with no version/resource/cache repair.
Do not retry a failed runtime under another run ID. Preserve evidence for review.

Bootstrap installs only `uv==0.8.22` with notebook pip `--target` into a fresh
dedicated directory and verifies `uv --version`, without creating a notebook-Python venv.
That uv obtains CPython **3.11.11**, then creates
a separate venv and installs the exact requirements. Notebook kernel packages are
not upgraded. Internet is used for repository/environment setup before the
separate exact-snapshot provision phase; runtime is entirely offline.

References: [D9R8 runbook](../notes/w2_paligemma_runner_smoke_prep.md),
[Kaggle settings](https://www.kaggle.com/docs/notebooks),
[uv Python installation](https://docs.astral.sh/uv/guides/install-python/),
[uv environments](https://docs.astral.sh/uv/pip/environments/).


In [ ]:
# Definitions only; no model/backend import or network request in this cell.
import os
import sys
if "torch" in sys.modules or "transformers" in sys.modules:
    raise RuntimeError("STOP: use a fresh notebook kernel before any backend import")
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import hashlib
import json
import math
import platform
import re
import shutil
import subprocess
import time
import zipfile
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath

BASE = "180c0623149bbc7d64f5b659f6c044c39b1e1cd0"
MODEL = "google/paligemma-3b-mix-448"
REVISION = "ead2d9a35598cb89119af004f5d023b311d1c4a1"
PLAN_HASH = "4138071ff3fcbcf2cb16d456c01a2b3dbc42a628b0faa0d04dd6a635803b0ab7"
WORKING = Path("/kaggle/working")  # Venue mount, not a personal machine path.
WORK = WORKING / "safeshift_d9r9"
REPO = WORK / "runtime_checkout"
SETUP = WORK / "evidence"
ENV = WORK / "runtime_env"
PYTHON = ENV / "bin/python"
CACHE = "data/processed/paligemma_d9r9_hf_cache"
MANIFESTS = "data/processed/d9r9_manifests"
RUN_ID = "paligemma-d9r9-01"  # Fixed attempt identity; never regenerate on failure.
RUN_REL = "data/processed/runtime_validation/w2_paligemma_t4_smoke/" + RUN_ID
BUNDLE = WORKING / "d9r9_result_bundle"
ZIP = WORKING / "d9r9_result_bundle.zip"
OFFLINE = {"HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1",
           "HF_HUB_DISABLE_TELEMETRY": "1", "HF_HUB_ENABLE_HF_TRANSFER": "0",
           "HF_HUB_DISABLE_XET": "1"}
SECRET_VALUES = []  # Memory only; never serialized or printed.
ONLINE_DONE = False
PHASE = "NOT_STARTED"
MAX_FILE = 4 * 1024**2
MAX_BUNDLE = 32 * 1024**2

def require(ok, message):
    if not ok:
        raise RuntimeError(message)

def digest(data):
    return hashlib.sha256(data).hexdigest()

def sanitize(text):
    for value in sorted(SECRET_VALUES, key=len, reverse=True):
        if value:
            text = text.replace(value, "[REDACTED]")
    text = re.sub(r"\b(?:hf_|ghp_|gho_|github_pat_)[A-Za-z0-9_]{16,}", "[REDACTED]", text)
    text = re.sub(r"(?i)(authorization\s*[:=]\s*)(?:bearer|basic)\s+\S+", r"\1[REDACTED]", text)
    text = re.sub(r"https?://[^\s/@]+:[^\s/@]+@", "https://[REDACTED]@", text)
    return text

def write_new(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    data = (json.dumps(value, indent=2, allow_nan=False) + "\n").encode()
    require(sanitize(data.decode()) == data.decode(), "STOP: secret-like content refused")
    with path.open("xb") as stream:
        stream.write(data)
        stream.flush()
        os.fsync(stream.fileno())

def child_env(offline=False):
    # Allowlist: never inherit notebook tokens, Python paths, Hub/Git credentials,
    # proxy credentials, or user pip settings into a child process.
    env = {key: os.environ[key] for key in
           ("PATH", "HOME", "LANG", "LC_ALL", "TMPDIR", "LD_LIBRARY_PATH") if key in os.environ}
    env.update(CUDA_DEVICE_ORDER="PCI_BUS_ID", CUDA_VISIBLE_DEVICES="0",
               PYTHONNOUSERSITE="1", PYTHONUNBUFFERED="1", PIP_CONFIG_FILE="/dev/null",
               HF_HOME=str(WORK / "private_hf_home"), HF_HUB_DISABLE_TELEMETRY="1",
               HF_HUB_ENABLE_HF_TRANSFER="0", HF_HUB_DISABLE_XET="1",
               GIT_TERMINAL_PROMPT="0", GIT_CONFIG_NOSYSTEM="1", GIT_CONFIG_GLOBAL="/dev/null",
               UV_PYTHON_INSTALL_DIR=str(WORK / "managed_python"),
               UV_PYTHON_BIN_DIR=str(WORK / "python_bin"), UV_CACHE_DIR=str(WORK / "uv_cache"))
    if offline:
        env.update(OFFLINE)
    return env

def command(args, label, *, cwd=None, env=None, failure=None, check=True):
    print("START", label, flush=True)
    started = time.perf_counter()
    result = subprocess.run([str(a) for a in args], cwd=cwd,
                            env=env if env is not None else child_env(),
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    # Sanitize before any disk write/display. Never log the environment.
    log = sanitize(result.stdout)
    require(len(log.encode()) <= MAX_FILE, "STOP: oversized log; no evidence truncation")
    with (SETUP / (label + ".log")).open("x", encoding="utf-8") as stream:
        stream.write(log)
    write_new(SETUP / (label + ".json"), {
        "returncode": result.returncode, "wall_seconds": time.perf_counter() - started})
    if check and result.returncode != 0:
        raise RuntimeError(failure or ("STOP: " + label + "; inspect sanitized log"))
    print("PASS" if result.returncode == 0 else "FAIL", label, flush=True)
    return result.returncode, log

def secret(name, required=True):
    value = None
    try:
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret(name)
    except Exception:
        pass  # Never propagate provider exceptions that could carry credentials.
    if not value:
        if required:
            raise RuntimeError("Create a Kaggle Secret named " + name + " and rerun.") from None
        return None
    SECRET_VALUES.append(value)
    return value

def clean_checkout():
    env = child_env(offline=True)
    for args, expected in ((["rev-parse", "HEAD"], BASE),
                           (["status", "--porcelain", "--untracked-files=all"], "")):
        value = subprocess.check_output(["git", *args], cwd=REPO, env=env, text=True).strip()
        require(value == expected, "STOP: exact clean execution checkout required")

def checked_file(root, relative):
    rel = PurePosixPath(relative)
    require(not rel.is_absolute() and ".." not in rel.parts and ":" not in relative
            and "\\" not in relative, "STOP: unsafe evidence path")
    path = root / relative
    require(path.resolve().is_relative_to(root.resolve()) and not path.is_symlink()
            and path.resolve() == path.absolute(), "STOP: evidence symlink/escape")
    require(path.is_file() and path.stat().st_size <= MAX_FILE, "STOP: missing/oversized evidence")
    return path

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def build_bundle(report):
    # Positive allowlist only. Never traverse the checkout/cache/environment.
    require(not BUNDLE.exists() and not ZIP.exists(), "STOP: bundle already exists; no overwrite")
    BUNDLE.mkdir()
    selected = []
    for path in sorted(SETUP.iterdir()):
        if path.suffix in {".json", ".log", ".py"}:
            selected.append((SETUP, path.name, "setup/" + path.name))
    manifests = REPO / MANIFESTS
    for name in ("provision.json", "verify_online.json", "verify_offline.json"):
        if (manifests / name).is_file():
            selected.append((manifests, name, "snapshot/" + name))
    runtime = REPO / RUN_REL
    if runtime.is_dir():
        allowed = {"summary.json", "summary.sha256.json", "run_metadata.json", "environment.json",
                   "snapshot_manifest.json", "case_01.png", "case_02.png"}
        for path in sorted(runtime.rglob("*")):
            if not path.is_file():
                continue
            rel = path.relative_to(runtime).as_posix()
            if rel in allowed or re.fullmatch(r"[0-9a-f]{64}/(?:response\.raw|metadata\.json|result\.json)", rel):
                selected.append((runtime, rel, "runtime/" + rel))
    inventory, total = {}, 0
    for root, relative, target in selected:
        path = checked_file(root, relative)
        raw = path.read_bytes()
        # Logs were sanitized at capture. Raw/model evidence is NEVER rewritten.
        if path.suffix != ".png":
            text = raw.decode("utf-8")
            require(sanitize(text) == text, "STOP: secret-like content in evidence; export refused")
        total += len(raw)
        require(total <= MAX_BUNDLE, "STOP: evidence bundle exceeds size limit")
        destination = BUNDLE / target
        destination.parent.mkdir(parents=True, exist_ok=True)
        with destination.open("xb") as stream:
            stream.write(raw)
        inventory[target] = {"sha256": digest(raw), "size_bytes": len(raw)}
    write_new(BUNDLE / "review_summary.json", report)
    raw = (BUNDLE / "review_summary.json").read_bytes()
    inventory["review_summary.json"] = {"sha256": digest(raw), "size_bytes": len(raw)}
    write_new(BUNDLE / "checksums.json", inventory)
    with zipfile.ZipFile(ZIP, "x", compression=zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(BUNDLE.rglob("*")):
            if path.is_file():
                archive.write(path, "d9r9_result_bundle/" + path.relative_to(BUNDLE).as_posix())
    print("BUNDLE:", ZIP)
    print("ZIP_SHA256:", digest(ZIP.read_bytes()))
    from IPython.display import FileLink, display
    display(FileLink(str(ZIP.relative_to(WORKING))))

def classify_timing(summary, timing):
    # The audited runner returns immediately when already loaded. Load count
    # comes from its independent summary, never from profiler span count/duration.
    require(summary["status"] == "RUNTIME_INTERFACE_PASS"
            and summary["model_load_count"] == 1 and summary["native_generate_calls"] == 2,
            "STOP: successful one-load/two-call summary required for timing interpretation")
    require(timing["observer_errors"] == [] and timing["unfinished_spans"] == 0,
            "STOP: timing observer errors or unfinished spans")
    measurements = timing["measurements"]
    load_scope = "runner_load_including_snapshot_and_transfer"
    require(bool(measurements) and measurements[0]["scope"] == load_scope,
            "STOP: initial actual load timing missing")
    for measurement in measurements:
        seconds = measurement["wall_seconds"]
        require(measurement["scope"] in {load_scope, "native_generate"}
                and type(seconds) in (int, float) and math.isfinite(seconds) and seconds >= 0,
                "STOP: invalid timing observation")
    loads = [m for m in measurements if m["scope"] == load_scope]
    generations = [m for m in measurements if m["scope"] == "native_generate"]
    require(len(generations) == 2, "STOP: exactly two native generation spans required")
    return {"basis": "SUCCESSFUL_D9R8_SUMMARY_MODEL_LOAD_COUNT_1_AND_IDEMPOTENT_LOAD_CONTRACT",
            "actual_load_span": loads[0], "idempotent_load_reentry_spans": loads[1:],
            "model_load_count": summary["model_load_count"],
            "load_seconds_including_snapshot_and_transfer": loads[0]["wall_seconds"],
            "call_generation_seconds": [m["wall_seconds"] for m in generations]}

def boundaries():
    return {"model": MODEL, "revision": REVISION, "execution_commit": BASE,
            "run_id": RUN_ID, "qualification_decision": "RESEARCH_LEAD_REVIEW_REQUIRED",
            "grounding_status": "PENDING_QUALIFICATION", "classification_interface_status": "PENDING_QUALIFICATION",
            "external_gate_status": "PENDING_QUALIFICATION", "classification_adapter_expected": "INVALID",
            "grounding_adapter_expected": "UNSUPPORTED", "grounding_adapter_executed": False,
            "synthetic_gate": "NOT_RUN", "inspecsafe": "NOT_RUN", "inspecsafe_authorized": False,
            "promotion": "NO", "paligemma_role": "BACKUP_1", "primary_roster_count": 4,
            "protocol_freeze": "BLOCKED"}


## Automatic online phase
This cell performs venue checks, fresh checkout, isolated install, exact software
and process-visible T4 checks, explicit provision, and independent verification.
Only the provision subprocess receives HF_TOKEN. Authentication failures do not
trigger anonymous retries. No model is loaded in this phase.


In [ ]:
require(platform.system() == "Linux" and platform.machine() == "x86_64"
        and WORKING.is_dir(), "STOP: Kaggle Linux x86_64 venue required")
require(not WORK.exists() and not BUNDLE.exists() and not ZIP.exists(),
        "STOP: dedicated workspace/evidence already exists; no repair, overwrite or automatic retry")
# Missing credential stops before creating a workspace, so the prescribed secret
# setup and rerun is safe. Other failures retain their workspace for review.
hf_token = secret("HF_TOKEN")
WORK.mkdir()
SETUP.mkdir()
try:
    PHASE = "VENUE"
    _, physical = command(["nvidia-smi", "--query-gpu=index,name,memory.total,compute_cap,driver_version",
                           "--format=csv,noheader,nounits"], "physical_gpu")
    rows = [[part.strip() for part in row.split(",")] for row in physical.strip().splitlines()]
    require(len(rows) in (1, 2), "STOP: expected one or two physical T4 GPUs")
    for index, name, memory, capability, driver in rows:
        require(name in {"T4", "Tesla T4", "NVIDIA T4"} and capability == "7.5"
                and 14 * 1024 <= float(memory) <= 16 * 1024, "STOP: physical GPU contract mismatch")
    write_new(SETUP / "venue.json", {"platform": platform.platform(), "physical_gpus": rows,
              "notebook_python": platform.python_version(), "free_disk_bytes": shutil.disk_usage(WORK).free,
              "cuda_visible_devices": "0", "cuda_device_order": "PCI_BUS_ID"})

    PHASE = "CHECKOUT"
    command(["git", "-c", "credential.helper=", "clone", "--no-checkout",
             "https://github.com/tantdna2/SafeShift.git", REPO], "clone",
            failure="STOP: anonymous public clone failed; preserve evidence for Research Lead review.")
    command(["git", "checkout", "--detach", BASE], "checkout", cwd=REPO)
    clean_checkout()
    plan = read_json(REPO / "configs/pre_freeze/paligemma_t4_runtime.v1.json")
    canonical = json.dumps(plan, sort_keys=True, separators=(",", ":"), ensure_ascii=True, allow_nan=False).encode()
    require(digest(canonical) == PLAN_HASH and plan["model_id"] == MODEL
            and plan["model_revision"] == REVISION and len(plan["snapshot_files"]) == 14,
            "STOP: pinned plan mismatch")
    write_new(SETUP / "execution_pin.json", {"execution_commit": BASE, "plan_sha256": PLAN_HASH,
              "requirements_sha256": digest((REPO / "requirements-paligemma-t4.txt").read_bytes())})

    PHASE = "SOFTWARE_INSTALL"
    bootstrap = WORK / "bootstrap"
    require(not bootstrap.exists(), "STOP: bootstrap location exists; no overwrite")
    command([sys.executable, "-m", "pip", "--isolated", "install", "--no-cache-dir",
             "--no-deps", "--only-binary=:all:", "--ignore-installed",
             "--target", bootstrap, "uv==0.8.22"], "bootstrap_uv")
    uv = bootstrap / "bin/uv"
    _, uv_version = command([uv, "--version"], "bootstrap_uv_version")
    require(re.fullmatch(r"uv 0\.8\.22(?: \([^\r\n()]+\))?", uv_version.strip()) is not None,
            "STOP: bootstrap requires exactly uv 0.8.22")
    command([uv, "--no-config", "python", "install", "3.11.11"], "python_install")
    command([uv, "--no-config", "venv", "--managed-python", "--python", "3.11.11", "--seed", ENV], "runtime_venv")
    command([PYTHON, "-m", "pip", "--isolated", "install", "--no-cache-dir", "--only-binary=:all:",
             "-r", "requirements-paligemma-t4.txt"], "install_pins", cwd=REPO,
            failure="STOP_AND_RESEARCH_LEAD_REVIEW_REQUIRED: exact package/wheel installation failed")
    command([PYTHON, "-m", "pip", "--isolated", "check"], "pip_check", cwd=REPO,
            failure="STOP_AND_RESEARCH_LEAD_REVIEW_REQUIRED: pip check failed")
    command([PYTHON, "-m", "pip", "--isolated", "list", "--format=json"], "installed_distributions", cwd=REPO)
    command([PYTHON, "-m", "pip", "--isolated", "freeze", "--all"], "pip_freeze", cwd=REPO)
    PHASE = "SOFTWARE_AND_PROCESS_GPU"
    probe = r'''import json
from safeshift.runners.paligemma_snapshot import load_plan, network_denied, require_offline_env
from safeshift.runners.paligemma import software_versions, probe_hardware
require_offline_env()
with network_denied():
    actual = software_versions()
    if actual != load_plan()["software"]:
        raise RuntimeError("STOP_AND_RESEARCH_LEAD_REVIEW_REQUIRED: exact pins mismatch")
    import torch
    hardware = probe_hardware(torch)
    print(json.dumps({"software": actual, "hardware": hardware}, indent=2))
'''
    _, probe_output = command([PYTHON, "-c", probe], "process_probe", cwd=REPO, env=child_env(offline=True))
    write_new(SETUP / "software_hardware.json", json.loads(probe_output))

    PHASE = "PROVISION"
    cache = REPO / CACHE
    require(not cache.exists(), "STOP: cache is not fresh; no repair or overwrite")
    provision_env = child_env()
    provision_env["HF_TOKEN"] = hf_token
    try:
        command([PYTHON, "scripts/provision_paligemma_snapshot.py", "--provision", "--cache-dir", CACHE,
                 "--manifest", MANIFESTS + "/provision.json"], "provision", cwd=REPO, env=provision_env)
    finally:
        provision_env.clear()
        hf_token = None
    PHASE = "VERIFY_SNAPSHOT_ONLINE"
    command([PYTHON, "scripts/provision_paligemma_snapshot.py", "--verify-only", "--cache-dir", CACHE,
             "--manifest", MANIFESTS + "/verify_online.json"], "verify_online", cwd=REPO, env=child_env(offline=True))
    verification = read_json(REPO / MANIFESTS / "verify_online.json")
    require(verification["local_bytes_verified"] is True and len(verification["files"]) == 14
            and verification["revision"] == REVISION, "STOP: snapshot verification not established")
    clean_checkout()
    write_new(SETUP / "online_complete.json", {"snapshot_verified": True, "execution_commit": BASE,
              "finished_at_utc": datetime.now(timezone.utc).isoformat()})
    ONLINE_DONE = True
    PHASE = "AWAITING_OWNER_INTERNET_OFF"
    print("SNAPSHOT_VERIFIED=true; PROVISION COMPLETE — MANUAL ACTION REQUIRED")
except Exception as exc:
    report = {**boundaries(), "status": "STOP_AND_REPORT", "failed_phase": PHASE,
              "reason": sanitize(str(exc)), "runtime_executed": False}
    write_new(SETUP / "failure.json", report)
    build_bundle(report)
    raise RuntimeError(report["reason"]) from None
finally:
    hf_token = None


---
## PROVISION COMPLETE — MANUAL ACTION REQUIRED

1. Kaggle -> Settings / Session options
2. **Turn Internet OFF**
3. Return to notebook
4. Set **OWNER_ATTEST_INTERNET_OFF = True** in the next cell
5. Continue from the next cell

**DO NOT restart the session. DO NOT rerun provisioning.**

This is the only interactive barrier. The notebook cannot turn off venue Internet.
The boolean is the owner's attestation, not a programmatic network measurement.
If Run All reaches the next cell while False, it stops before any runtime attempt.

The next cell automatically verifies again offline, executes the unchanged harness,
checks artifacts, records OBSERVED_ONLY native output and exports a small bundle.
It never grants research qualification. On failure it collects available evidence
and stops without retry. A killed kernel may prevent final export; retain the
workspace for Research Lead review, and do not restart/reinterpret the run.

**Timing scope:** D9R8 does not expose durations. A stdlib `sys.setprofile` observer
executes its exact CLI via `runpy` without changing model/runner functions. It records
the runner's `load` span (including snapshot verification and device transfer),
and native `GenerationMixin.generate` spans with CUDA synchronization at entry/exit.
These are instrumented wall times, not optimization benchmarks.
Load re-entry spans are recorded separately as idempotent/no-op only when the
independent successful runner summary confirms model_load_count=1. Exactly two
native generation spans are required; load span count is not model load count.
Failure/unreached spans remain absent/null; durations are never fabricated. The observer source is
included in the bundle. Raw persistence and adaptation remain owned by D9R8.


In [ ]:
OWNER_ATTEST_INTERNET_OFF = False
require(OWNER_ATTEST_INTERNET_OFF is True,
        "STOP: turn venue Internet OFF, set OWNER_ATTEST_INTERNET_OFF = True, then continue this cell")
require(ONLINE_DONE is True and (SETUP / "online_complete.json").is_file(),
        "STOP: verified online phase must finish in this session")
require(not (SETUP / "runtime_attempt.json").exists(),
        "STOP: runtime attempt already started; no retry or fresh call_id")
write_new(SETUP / "runtime_attempt.json", {"owner_attested_internet_off": True,
          "attested_at_utc": datetime.now(timezone.utc).isoformat(), "run_id": RUN_ID})
runtime_env = child_env(offline=True)
OBSERVER = 'import json\nimport runpy\nimport sys\nimport time\nfrom pathlib import Path\n\nrepo = Path.cwd()\nsys.path.insert(0, str(repo))\nfrom safeshift.runners.paligemma_snapshot import network_denied, require_offline_env\n\nactive, measurements, errors = {}, [], []\n\ndef observe(frame, event, arg):\n    filename = frame.f_code.co_filename.replace("\\\\", "/")\n    name = frame.f_code.co_name\n    kind = None\n    if filename.endswith("/safeshift/runners/paligemma.py") and name == "load":\n        kind = "runner_load_including_snapshot_and_transfer"\n    elif filename.endswith("/transformers/generation/utils.py") and name == "generate":\n        kind = "native_generate"\n    if kind is None or event not in {"call", "return"}:\n        return\n    try:\n        torch = sys.modules.get("torch")\n        if torch is not None:\n            torch.cuda.synchronize(0)\n        if event == "call":\n            active[id(frame)] = (kind, time.perf_counter())\n        elif id(frame) in active:\n            label, start = active.pop(id(frame))\n            measurements.append({"scope": label, "wall_seconds": time.perf_counter() - start})\n    except Exception as exc:\n        # Do not alter inference control flow. Any timing error prevents notebook completion.\n        errors.append(type(exc).__name__)\n\ntiming_path = Path(sys.argv[1])\nsys.argv = ["scripts/w2_paligemma_t4_smoke.py", *sys.argv[2:]]\nstarted = time.perf_counter()\ntry:\n    require_offline_env()\n    with network_denied():\n        sys.setprofile(observe)\n        runpy.run_path(sys.argv[0], run_name="__main__")\nfinally:\n    sys.setprofile(None)\n    with timing_path.open("x", encoding="utf-8") as stream:\n        json.dump({"measurements": measurements, "observer_errors": errors,\n                   "unfinished_spans": len(active), "harness_wall_seconds": time.perf_counter() - started,\n                   "instrumentation": "sys.setprofile; CUDA synchronization; unchanged D9R8 CLI via runpy"}, stream, indent=2)\n'
try:
    PHASE = "OFFLINE_VERIFY"
    clean_checkout()
    command([PYTHON, "scripts/provision_paligemma_snapshot.py", "--verify-only", "--cache-dir", CACHE,
             "--manifest", MANIFESTS + "/verify_offline.json"], "verify_offline", cwd=REPO, env=runtime_env)
    # Keep orchestration outside the exact runtime checkout, and include it in evidence.
    observer_path = SETUP / "timing_observer.py"
    with observer_path.open("x", encoding="utf-8") as stream:
        stream.write(OBSERVER)
    PHASE = "REAL_SMOKE"
    clean_checkout()
    returncode, _ = command([PYTHON, observer_path, SETUP / "timings.json",
                            "--run-id", RUN_ID, "--expected-commit", BASE,
                            "--cache-dir", CACHE, "--venue-internet-off"],
                           "smoke", cwd=REPO, env=runtime_env, check=False)
    PHASE = "ARTIFACT_REVIEW"
    root = REPO / RUN_REL
    summary_path = checked_file(root, "summary.json")
    summary_bytes = summary_path.read_bytes()
    require(digest(summary_bytes) == read_json(checked_file(root, "summary.sha256.json"))["sha256"],
            "STOP: summary checksum mismatch")
    summary = json.loads(summary_bytes)
    for relative, ref in summary["artifacts"].items():
        raw = checked_file(root, relative).read_bytes()
        require(len(raw) == ref["size_bytes"] and digest(raw) == ref["sha256"],
                "STOP: evidence checksum/size mismatch")
    metadata = read_json(checked_file(root, "run_metadata.json"))
    environment = read_json(checked_file(root, "environment.json"))
    snapshot = read_json(checked_file(root, "snapshot_manifest.json"))
    timing = read_json(SETUP / "timings.json")
    calls = []
    for call in summary["calls"]:
        ref = call.get("raw")
        observation = None
        if ref:
            prefix = RUN_REL + "/"
            require(ref["path"].startswith(prefix), "STOP: raw outside this run")
            raw = checked_file(root, ref["path"][len(prefix):]).read_bytes()
            require(digest(raw) == ref["sha256"] and len(raw) == ref["size_bytes"],
                    "STOP: raw checksum/size mismatch")
            envelope = json.loads(raw)  # Only after D9R8 persistence and checksum verification.
            raw_meta = read_json(checked_file(root, ref["path"][len(prefix):].rsplit("/", 1)[0] + "/metadata.json"))
            require(raw_meta["parse_status"] == "NOT_ATTEMPTED" and raw_meta["call_id"] == call["case_id"]
                    and raw_meta["git_commit_sha"] == BASE, "STOP: raw-before-adapter provenance mismatch")
            observation = {"status": "OBSERVED_ONLY", "generated_ids_full": envelope.get("generated_ids_full"),
                           "continuation_ids": envelope.get("continuation_ids"),
                           "decoded_with_special_tokens": envelope.get("decoded_with_special_tokens"),
                           "decoded_text": envelope.get("decoded_text"),
                           "loc_token_text_present": "<loc" in envelope.get("decoded_with_special_tokens", ""),
                           "classification_grammar": "RESEARCH_LEAD_OBSERVATION_REQUIRED_NO_PRODUCTION_GRAMMAR_FROZEN",
                           "multiple_box_separators": "NOT_TESTED_BY_COLOR_QUESTION",
                           "no_detection_behavior": "NOT_TESTED_BY_COLOR_QUESTION"}
        calls.append({"call_id": call["case_id"], "adapter_status": call["parse_status"],
                      "raw": ref, "observation": observation, "error": call.get("error")})
    report = {**boundaries(), "status": summary["status"], "harness_exit_code": returncode,
              "snapshot_verified": snapshot.get("local_bytes_verified", False),
              "hardware": environment.get("hardware"), "software": environment.get("software"),
              "offline_env": environment.get("offline_variables"), "owner_attested_internet_off": True,
              "model_load_count": summary["model_load_count"], "native_generate_calls": summary["native_generate_calls"],
              "model_load_status": "COMPLETED" if "after_load" in summary["memory"] else "NOT_ESTABLISHED",
              "loading_info_status": "PASSED_D9R8_CHECK" if "after_load" in summary["memory"] else "NOT_ESTABLISHED",
              "calls": calls, "memory": summary["memory"], "timings": timing,
              "failure": summary.get("failure"), "artifact_checksums_verified": True,
              "raw_before_adapter": "D9R8_VERIFIED_PIPELINE_WITH_NOT_ATTEMPTED_PROVENANCE"
              if len(calls) == 2 and all(c["raw"] for c in calls) else "NOT_ESTABLISHED"}
    require(returncode == 0 and summary["status"] == "RUNTIME_INTERFACE_PASS",
            "STOP: runtime failed; see preserved D9R8 evidence; no retry")
    require(metadata["git_commit"] == BASE and metadata["venue_internet_off_owner_attested"] is True
            and snapshot["local_bytes_verified"] is True and len(snapshot["files"]) == 14,
            "STOP: runtime provenance mismatch")
    require(environment["offline_variables"] == OFFLINE
            and environment["software"] == plan["software"]
            and environment["hardware"]["visible_gpu_count"] == 1
            and environment["hardware"]["gpu_name"] in {"T4", "Tesla T4", "NVIDIA T4"}
            and environment["hardware"]["compute_capability"] == [7, 5]
            and 14 * 2**30 <= environment["hardware"]["total_vram_bytes"] <= 16 * 2**30
            and environment["hardware"]["cuda_runtime"] == "12.4"
            and environment["hardware"]["device"] == "cuda:0",
            "STOP: recorded runtime environment mismatch")
    require(summary["model_load_count"] == 1 and summary["native_generate_calls"] == 2
            and [c["call_id"] for c in calls] == ["case_01", "case_02"]
            and all(c["adapter_status"] == "INVALID" and c["raw"] for c in calls)
            and len({c["raw"]["path"] for c in calls}) == 2,
            "STOP: one-load/two-independent-call contract mismatch")
    timing_analysis = classify_timing(summary, timing)
    clean_checkout()
    report["runtime_contract_checks"] = "PASS_RECORDED_FACTS_ONLY"
    report["peak_gpu_memory"] = {key: max(m.get(key, 0) for m in summary["memory"].values())
                                 for key in ("peak_allocated_bytes", "peak_reserved_bytes")}
    report["timing_analysis"] = timing_analysis
    report["load_time_seconds_including_snapshot_and_transfer"] = timing_analysis["load_seconds_including_snapshot_and_transfer"]
    report["call_generation_seconds"] = timing_analysis["call_generation_seconds"]
except Exception as exc:
    report = {**boundaries(), "status": "STOP_AND_REPORT", "failed_phase": PHASE,
              "reason": sanitize(str(exc)), "runtime_attempt_recorded": True,
              "evidence": "Inspect unchanged runtime/summary.json and setup logs if present"}
    write_new(SETUP / "failure.json", report)
    build_bundle(report)
    raise RuntimeError(report["reason"]) from None
else:
    build_bundle(report)
    print(json.dumps(report, indent=2))
    print("D9R9_NOTEBOOK_EXECUTION_COMPLETE")
    print(ZIP)
finally:
    runtime_env.clear()
    SECRET_VALUES.clear()


The completion marker means orchestration and evidence export finished. It does
**not** mean Research Lead qualification, grounding PASS, promotion or protocol
freeze. Submit `d9r9_result_bundle.zip` to the Research Lead. Native text and IDs
are observations only; do not modify the runner/parser based on these two calls.
If observations contradict documentary assumptions, stop for Research Lead review.
No synthetic-v1 or InspecSafe execution is included.
